<a href="https://colab.research.google.com/github/shivashankarb2006/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shivashankarb2006/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Ranked actions + reason codes

The action queue prioritizes content pages for human review using observed historical performance signals.

Each recommendation has:
- a priority score,
- an action,
- and one reason code explaining the main signal behind the recommendation.

Reason codes:

- DECLINING_TRAFFIC: recent impressions are substantially below the previous period.
- LOW_QUERY_COVERAGE: the page has relatively few visible queries.
- HIGH_QUERY_CONCENTRATION: impressions are concentrated in a small number of queries.
- STABLE_MONITOR: the available signals do not justify an immediate optimization action.

The queue is a prioritization tool, not an automatic instruction to change a page.

In [5]:
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 283 (delta 112), reused 83 (delta 83), pack-reused 138 (from 1)
Receiving objects: 100% (283/283), 1.85 MiB | 5.20 MiB/s, done.
Resolving deltas: 100% (153/153), done.


In [6]:
import pandas as pd
import numpy as np

# Load the starter dataset
df = pd.read_csv(
    "flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"
)

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [7]:
data = df.copy()

# Create the previous and recent impression fields
data['imp_prev30'] = data['impressions_prev_30d']
data['imp_last30'] = data['impressions_last_30d']

# Create query-related proxy features
data['visible_queries'] = data['search_volume']

# Use available performance signals as proxies
data['top_query_share'] = data['ctr']

# Create the target used in the earlier model
data['is_declining'] = (
    data['imp_last30'] < 0.8 * data['imp_prev30']
).astype(int)

print("Data prepared:", data.shape)

data[
    [
        'content_id',
        'imp_prev30',
        'imp_last30',
        'visible_queries',
        'top_query_share',
        'is_declining'
    ]
].head()

Data prepared: (30000, 49)


,content_id,imp_prev30,imp_last30,visible_queries,top_query_share,is_declining
0,content_304f48230142,987,578,10.0,0.76,1
1,content_a1fb4e703a9e,5915,2501,90.0,0.05,1
2,content_9aa793d4d895,6089,2382,0.0,0.09,1
3,content_331d6c4de07b,4206,3626,10.0,0.49,0
4,content_d99b7a2d90ca,6452,4211,0.0,0.13,1


In [9]:
queue = data.copy()

queue['decline_pct'] = np.where(
    queue['imp_prev30'] > 0,
    (queue['imp_prev30'] - queue['imp_last30'])
    / queue['imp_prev30'],
    0
)

queue['priority_score'] = (
    queue['decline_pct'].clip(lower=0) * 60
    + queue['top_query_share'].fillna(0) * 25
    + (
        1 / queue['visible_queries'].clip(lower=1)
    ).fillna(0) * 15
)

queue['reason_code'] = np.select(
    [
        queue['decline_pct'] >= 0.20,
        queue['visible_queries'] <= 5,
        queue['top_query_share'] >= 0.50
    ],
    [
        'DECLINING_TRAFFIC',
        'LOW_QUERY_COVERAGE',
        'HIGH_QUERY_CONCENTRATION'
    ],
    default='STABLE_MONITOR'
)

queue['action'] = np.select(
    [
        queue['reason_code'] == 'DECLINING_TRAFFIC',
        queue['reason_code'] == 'LOW_QUERY_COVERAGE',
        queue['reason_code'] == 'HIGH_QUERY_CONCENTRATION'
    ],
    [
        'REVIEW_AND_REFRESH',
        'REVIEW_QUERY_COVERAGE',
        'REVIEW_CONTENT_SCOPE'
    ],
    default='MONITOR'
)

queue = queue.sort_values(
    'priority_score',
    ascending=False
).reset_index(drop=True)

queue['rank'] = np.arange(1, len(queue) + 1)

print("Queue created:", len(queue))

queue[
    [
        'rank',
        'content_id',
        'priority_score',
        'reason_code',
        'action'
    ]
].head(20)

Queue created: 30000


,rank,content_id,priority_score,reason_code,action
0,1,content_a8cee66e4788,2575.00,DECLINING_TRAFFIC,REVIEW_AND_REFRESH
1,2,content_3f3576c295f5,2515.00,LOW_QUERY_COVERAGE,REVIEW_QUERY_COVERAGE
2,3,content_4272d3a330a3,2515.00,LOW_QUERY_COVERAGE,REVIEW_QUERY_COVERAGE
3,4,content_006b16e7a2e7,2515.00,LOW_QUERY_COVERAGE,REVIEW_QUERY_COVERAGE
4,5,content_6016b918a48f,2500.00,HIGH_QUERY_CONCENTRATION,REVIEW_CONTENT_SCOPE
5,6,content_a84e013a5f94,2500.00,HIGH_QUERY_CONCENTRATION,REVIEW_CONTENT_SCOPE
6,7,content_cfa4d9f1bf0a,2500.00,HIGH_QUERY_CONCENTRATION,REVIEW_CONTENT_SCOPE
7,8,content_98458bafe297,2500.00,HIGH_QUERY_CONCENTRATION,REVIEW_CONTENT_SCOPE
8,9,content_bf398aa7400e,2500.00,HIGH_QUERY_CONCENTRATION,REVIEW_CONTENT_SCOPE
9,10,content_bc2c0c7243df,2500.00,HIGH_QUERY_CONCENTRATION,REVIEW_CONTENT_SCOPE


## Intended use and limits

The playbook is intended for content teams and SEO analysts who need to decide which pages deserve human review first.

The output is decision-support based on observed historical signals. A high-ranked page should be investigated before any content change is made.

The playbook is not valid as an automatic content-editing system. It does not establish that a page needs a rewrite, prove the cause of a traffic change, or predict future search-engine behavior.

The recommendations may become less reliable when the underlying search environment, content mix, measurement system, or client population changes substantially.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Human review + the no-go list

Before acting on a recommendation, a person should check:

1. Whether the recent traffic change is meaningful and not caused by a measurement issue.
2. Whether the page is still relevant to its intended search intent.
3. Whether the page has important business or informational value.
4. Whether the recommendation is supported by more than one signal.
5. Whether recent changes have already been made to the page.
6. Whether the proposed change could reduce useful existing traffic.

### No-go list

The system should never automatically:

- delete a page,
- publish or rewrite content,
- change important business information,
- claim that a ranking change was caused by a specific factor,
- claim to predict Google's ranking algorithm,
- make irreversible content decisions without human review.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Monitoring / retrain triggers

The recommendations should be reviewed when the data distribution changes substantially.

Potential triggers include:

- a large change in the proportion of declining pages,
- major changes in the distribution of impressions or query counts,
- new or missing feature values,
- changes in the content population,
- deterioration in validation performance,
- repeated disagreement between recommendations and human review.

The model or scoring system should be reconsidered and potentially retrained when these changes persist rather than assuming that historical relationships remain valid indefinitely.

In [13]:
print("===== MONITORING CHECKS =====")

print("Rows:", len(queue))

print("\nDeclining rate:")
print(round(queue['is_declining'].mean(), 3))

# Define the features used by the model
feature_cols = [
    'imp_prev30',
    'visible_queries',
    'rare_share',
    'anon_share',
    'top_query_share',
    'query_diversity'
]

print("\nMissing feature rates:")
for col in feature_cols:
    if col in queue.columns:
        print(col, round(queue[col].isna().mean(), 3))
    else:
        print(col, "COLUMN NOT FOUND")

print("\nReason-code distribution:")
print(queue['reason_code'].value_counts())

===== MONITORING CHECKS =====
Rows: 30000

Declining rate:
0.542

Missing feature rates:
imp_prev30 0.0
visible_queries 0.082
rare_share COLUMN NOT FOUND
anon_share COLUMN NOT FOUND
top_query_share 0.0
query_diversity COLUMN NOT FOUND

Reason-code distribution:
reason_code
DECLINING_TRAFFIC           16305
STABLE_MONITOR               8257
LOW_QUERY_COVERAGE           4052
HIGH_QUERY_CONCENTRATION     1386
Name: count, dtype: int64


## Exports for the paper

The ranked queue is exported so that the final research paper can reuse the same recommendations produced by this notebook.

The exported queue contains the rank, priority score, reason code, action, and supporting signals.

The export is intended for reproducibility and review. It does not represent an automatic decision system.

In [15]:
import os

os.makedirs("work/outputs", exist_ok=True)

paper_queue = queue[
    [
        'rank',
        'content_id',
        'priority_score',
        'reason_code',
        'action',
        'decline_pct',
        'imp_prev30',
        'imp_last30',
        'visible_queries',
        'top_query_share'
    ]
].copy()

output_path = "work/outputs/action_playbook_queue.csv"

paper_queue.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows exported:", len(paper_queue))

paper_queue.head(20)

Saved: work/outputs/action_playbook_queue.csv
Rows exported: 30000


,rank,content_id,priority_score,reason_code,action,decline_pct,imp_prev30,imp_last30,visible_queries,top_query_share
0,1,content_a8cee66e4788,2575.00,DECLINING_TRAFFIC,REVIEW_AND_REFRESH,1.0,1,0,0.0,100.00
1,2,content_3f3576c295f5,2515.00,LOW_QUERY_COVERAGE,REVIEW_QUERY_COVERAGE,0.0,0,0,0.0,100.00
2,3,content_4272d3a330a3,2515.00,LOW_QUERY_COVERAGE,REVIEW_QUERY_COVERAGE,0.0,0,0,0.0,100.00
3,4,content_006b16e7a2e7,2515.00,LOW_QUERY_COVERAGE,REVIEW_QUERY_COVERAGE,0.0,0,0,0.0,100.00
4,5,content_6016b918a48f,2500.00,HIGH_QUERY_CONCENTRATION,REVIEW_CONTENT_SCOPE,0.0,0,0,NaN,100.00
5,6,content_a84e013a5f94,2500.00,HIGH_QUERY_CONCENTRATION,REVIEW_CONTENT_SCOPE,0.0,0,0,NaN,100.00
6,7,content_cfa4d9f1bf0a,2500.00,HIGH_QUERY_CONCENTRATION,REVIEW_CONTENT_SCOPE,0.0,0,1,NaN,100.00
7,8,content_98458bafe297,2500.00,HIGH_QUERY_CONCENTRATION,REVIEW_CONTENT_SCOPE,0.0,0,0,NaN,100.00
8,9,content_bf398aa7400e,2500.00,HIGH_QUERY_CONCENTRATION,REVIEW_CONTENT_SCOPE,0.0,0,0,NaN,100.00
9,10,content_bc2c0c7243df,2500.00,HIGH_QUERY_CONCENTRATION,REVIEW_CONTENT_SCOPE,0.0,0,0,NaN,100.00


In [16]:
print(queue.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'imp_prev30', 'imp_last30', 'visible_queries', 'top_query_share', 'is_declining', 'decline_pct', 'priority_score', 'reason_code', 'action', 'rank']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.